In [4]:
import re
from typing import Tuple, List


# ------------------------------------------------------
#               PRIVACY FILTER
# ------------------------------------------------------
class PrivacyFilter:
    """Detect and redact emails, phone numbers, credit card numbers, and addresses."""
    def __init__(self):
        self.patterns = [
            # Email
            (re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'), "[EMAIL]"),

            # Credit card
            (re.compile(r'\b(?:\d[ -]*?){13,16}\b'), "[CARD]"),

            # Phone numbers
            (re.compile(r'\b(?:\+?1[-.\s]?)?\(?(\d{3})\)?[-.\s]?(\d{3})[-.\s]?(\d{4})\b'), "[PHONE]"),

            # Addresses (simple patterns)
            (re.compile(
                r'\b(\d{1,5}\s+(?:[A-Za-z0-9]+\s){0,4}(Street|St|Road|Rd|Avenue|Ave|Lane|Ln|Block|House|Way))\b',
                re.IGNORECASE),
                "[ADDRESS]"
            )
        ]

    def redact(self, text: str) -> Tuple[str, dict]:
        report = {}
        for regex, replacement in self.patterns:
            matches = regex.findall(text)

            if matches:
                clean_list = []
                for m in matches:
                    if isinstance(m, tuple):
                        clean_list.append(" ".join(m))
                    else:
                        clean_list.append(m)

                report[replacement] = list(set(clean_list))

            text = regex.sub(replacement, text)
        return text, report


# ------------------------------------------------------
#               MODERATION FILTER
# ------------------------------------------------------
class ModerationFilter:
    """Detect slang, profanity, and adult words."""
    def __init__(self, bad_words=None, replace_with="[REDACTED]"):
        self.bad_words = bad_words or ["asshole", "shit", "fuck", "bitch", "bastard"]
        self.replace_with = replace_with

        # Build regex patterns
        self.bad_word_patterns = [
            re.compile(self._build_pattern(word), re.IGNORECASE)
            for word in self.bad_words
        ]

    def _build_pattern(self, word: str) -> str:
        pattern = r"\b"
        for c in word:
            pattern += re.escape(c) + r"[\W_]*"
        return pattern + r"\b"

    def normalize(self, text: str) -> str:
        table = str.maketrans({'$': 's', '@': 'a', '1': 'l', '0': 'o', '3': 'e', '!': 'i'})
        return text.translate(table)

    def moderate(self, text: str) -> Tuple[str, List[str]]:
        found = []
        normalized = self.normalize(text)
        out = text

        for pattern in self.bad_word_patterns:
            for m in pattern.finditer(normalized):
                letters = re.sub(r"[^A-Za-z]", "", m.group())

                if not letters:
                    continue

                real_pattern = re.compile(r"(" + r"[\W_]*".join(list(letters)) + r")", re.IGNORECASE)

                def replacer(match):
                    found.append(match.group(0))
                    return self.replace_with

                out = real_pattern.sub(replacer, out)

        return out, list(set(found))


# ------------------------------------------------------
#               LLM SAFETY DETECTOR
# ------------------------------------------------------
def call_llm(prompt: str) -> str:
    """LLM detects unsafe content & returns safe output."""

    danger_words = {
        "violence": ["kill", "murder", "attack", "bomb"],
        "self_harm": ["suicide", "hurt myself", "die", "cut myself"],
        "sexual": ["sex", "nude", "porn","Fuck"],
        "hate": ["hate muslims", "racist", "kill jews"],
        "illegal": ["drug recipe", "hack server", "fake passport"]
    }

    # Safety detection
    prompt_low = prompt.lower()

    for category, words in danger_words.items():
        for w in words:
            if w in prompt_low:
                return f"⚠️ Unsafe request detected ({category}). I cannot help with this topic."

    # Normal response
    return f"LLM answer: {prompt}"


# ------------------------------------------------------
#               PIPELINE
# ------------------------------------------------------
class AgentPipeline:
    """Privacy → Moderation → LLM Response."""
    def __init__(self):
        self.privacy = PrivacyFilter()
        self.moderation = ModerationFilter()

    def process(self, text: str):
        # Step 1 — Privacy
        priv_clean, priv_report = self.privacy.redact(text)

        # Step 2 — Moderation
        mod_clean, mod_report = self.moderation.moderate(priv_clean)

        # Step 3 — LLM Safety + Response
        llm_response = call_llm(mod_clean)

        return mod_clean, {"privacy": priv_report, "moderation": mod_report}, llm_response


# ------------------------------------------------------
#               DEMONSTRATION
# ------------------------------------------------------
if __name__ == "__main__":
    pipeline = AgentPipeline()

    tests = [
        "My email is test@mail.com and card is 4111 1111 1111 1111. I live at 120 Baker Street.",
        "You are a f@#king a$$hole! Shit man!",
        "What is the capital of Bangladesh ?",
    ]

    for t in tests:
        cleaned, report, llm = pipeline.process(t)

        print("---- INPUT ----")
        print(t)
        print("---- CLEANED ----")
        print(cleaned)
        print("---- REPORT ----")
        print(report)
        print("---- LLM ----")
        print(llm)
        print()


---- INPUT ----
My email is test@mail.com and card is 4111 1111 1111 1111. I live at 120 Baker Street.
---- CLEANED ----
My email is [EMAIL] and card is [CARD]. I live at [ADDRESS].
---- REPORT ----
{'privacy': {'[EMAIL]': ['test@mail.com'], '[CARD]': ['4111 1111 1111 1111'], '[ADDRESS]': ['120 Baker Street Street']}, 'moderation': []}
---- LLM ----
LLM answer: My email is [EMAIL] and card is [CARD]. I live at [ADDRESS].

---- INPUT ----
You are a f@#king a$$hole! Shit man!
---- CLEANED ----
You are a f@#king a$$hole! [REDACTED] man!
---- REPORT ----
{'privacy': {}, 'moderation': ['Shit']}
---- LLM ----
LLM answer: You are a f@#king a$$hole! [REDACTED] man!

---- INPUT ----
What is the capital of Bangladesh ?
---- CLEANED ----
What is the capital of Bangladesh ?
---- REPORT ----
{'privacy': {}, 'moderation': []}
---- LLM ----
LLM answer: What is the capital of Bangladesh ?

